**Assessment Criteria** (Prioritized from most to least important):

1. Utilize a GPU to fine-tune a pre-trained model (must be no larger than **8B** for example I recomend the Qwen3-8B; only **one epoch** of fine-tuning required, I suggest using QLoRA(If u use 4 bit, maybe no more than 10GB GPU you will use)) on a specified dataset to achieve the highest possible accuracy on the test dataset. Please show off your tuning skills.
2. Showcase and summarize your results and key findings.
3. Conduct an analysis of the training/test dataset.
4. Provide clear explanations for the chosen functions and parameters, and how you achieve these hyper-parameters.
5. Complete all tasks as specified in the provided comments.

**Deadline for Submission**: 

**Submission Guidelines** - Please include the following in your submission email (Note: Do not reply to all):
- The link to your Colab notebook.
- A document detailing your tuning strategy, methods for improving accuracy, and explanations for why your approach was successful, etc.

Good luck and this is just a test that let me know if you have the ability to finish a Fintune Task so that I will know what we could do together ~ So take it easy

No need to use the Fintune framework below, it is just a reference, u could use every method you want, for example, I suggest unsloth, because it is much easier than other frame work~

In [ ]:
# target_model
target_model = ""# pick a llm model from huggingface

test_path='/content/...'
train_path='/content/...'

In [ ]:
%%capture
!pip install -U bitsandbytes
!pip install transformers==4.36.2
!pip install -U peft
!pip install -U accelerate
!pip install -U trl
!pip install datasets==2.16.0
!pip install sentencepiece

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,HfArgumentParser,TrainingArguments,pipeline, logging

import re
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import os,torch
from datasets import load_dataset
from trl import SFTTrainer
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pandas as pd
from datasets import Dataset


# Login to HF

In [ ]:
secret_hf = "..."
!huggingface-cli login --token $secret_hf

# Build Dataset

In [ ]:
# Analysis the data, Preprocess data, and Generate a report

df = pd.read_csv()
df['text'] = '...'


dataset = Dataset(pa.Table.from_pandas(df))
dataset

# Load the model

In [ ]:
# Load base model, use qlora
# Report the meaning of different parameters, and how they affect the training
bnb_config = BitsAndBytesConfig(

)
model = AutoModelForCausalLM.from_pretrained(

)


model.config.use_cache = False # silence the warnings. Please re-enable for inference!
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained()


In [ ]:
# count trainging statistics of training tokens
# Plot graphs if possible


# Fine-Tune

In [ ]:
# Add the adapters

model = get_peft_model(model, peft_config)

In [ ]:
# Set hyperparamters
# set them carefully, report the different effects
# just train 1 epoch is ok
training_arguments = TrainingArguments(

)


In [ ]:
# Setting sft parameters
trainer = SFTTrainer(

)

In [ ]:
trainer.train()

In [ ]:
# Plot loss graph, optionally gradient graph

# Save the model

In [ ]:
# Save the fine-tuned model


# Test the model

In [ ]:
# DON' CHANGE the code below. (Report if there are bugs)
def test_model(test_path, max_fail_cnt = 10):
    pipe = pipeline(task="text-generation")
    def build_prompt(question):
        prompt=f"<s>[INST] {question} [/INST]"
        return prompt

    df_test = pd.read_csv(test_path)
    correct_answers = 0
    question_counter = 0

    for index, row in df_test.iterrows():
        print("#############################")
        question_counter += 1

        question = 'Answer the question, write the answer like this at the end of your response: Answer:a or Answer:b or Answer:c or Answer:d \n'
        question += row['Question'] + '\na)' + row['a'] + '\nb)' + row['b'] + '\nc)' + row['c'] + '\nd)' + row['d'] + '\n'
        print(question)

        truth = row['Answer']
        index = -1
        fail_counter = 0

        while index == -1:
            prompt = build_prompt(question)
            result = pipe(prompt)
            llm_answer = result[0]['generated_text']
            index = llm_answer.find("[/INST]")
            llm_answer = llm_answer[index + len("[/INST]"):]
            llm_answer = llm_answer.replace(' ', '')
            index = llm_answer.find('Answer:')

            if index + len('Answer:') == len(llm_answer):
                index = -1

            fail_counter += 1

            if fail_counter == max_fail_cnt:
                break

        if fail_counter == max_fail_cnt:
            continue

        next_char = llm_answer[index + len('Answer:'):][0]

        if next_char in truth:
            correct_answers += 1

    accuracy = correct_answers / question_counter
    print(f"Total Questions: {question_counter}")
    print(f"Correct Answers: {correct_answers}")
    print(f"Accuracy: {accuracy}")

# Give a interactive script of how as a user I can use your product

In [ ]:
# Interactive UI

--------------------------